In [3]:
import pandas as pd
import os

# --- CONFIGURACIÓN DE RUTAS ---
RADIOMICS_DIR = "/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/radiomic_results/"
PFIRRMANN_CSV = "/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/code/total_segmentator/updated_patients_per_discs.csv"
OUTPUT_CSV = "/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/radiomics_final_with_labels.csv"

# Mapeo idéntico al usado en la segmentación (1=L5-S1, ..., 5=L1-L2)
DISC_MAP = {
    1: "L5-S1",
    2: "L4-L5",
    3: "L3-L4",
    4: "L2-L3",
    5: "L1-L2"
}

def combine_radiomics_and_labels():
    # 1. Cargar y unir los 5 CSVs de radiómica
    all_radiomics = []
    
    for i in range(1, 6):
        file_path = os.path.join(RADIOMICS_DIR, f"features_T2_mask_{i}.csv")
        if os.path.exists(file_path):
            df_temp = pd.read_csv(file_path)
            # Añadimos la columna 'disc' para identificar el origen anatómico
            df_temp['disc'] = DISC_MAP[i]
            all_radiomics.append(df_temp)
            print(f"Cargado disco {i} ({DISC_MAP[i]})")
        else:
            print(f"Aviso: No se encontró {file_path}")

    if not all_radiomics:
        print("Error: No se cargaron archivos de radiómica.")
        return

    df_radiomics = pd.concat(all_radiomics, ignore_index=True)

    # 2. Cargar el CSV de etiquetas Pfirrmann
    df_labels = pd.read_csv(PFIRRMANN_CSV)
    df_labels_subset = df_labels[['patient_id', 'disc', 'Pfirrmann']]

    # 3. Realizar el MERGE (Unión) por paciente y disco
    df_final = pd.merge(
        df_radiomics, 
        df_labels_subset, 
        on=['patient_id', 'disc'], 
        how='inner'
    )

    # 4. ORDENAR ALFABÉTICAMENTE POR patient_id
    # También ordenamos por 'disc' para que dentro de cada paciente los discos sigan un orden lógico
    df_final = df_final.sort_values(by=['patient_id', 'disc']).reset_index(drop=True)

    # 5. Reorganizar columnas para mejor visibilidad
    cols = list(df_final.columns)
    # Ponemos la información clave al principio
    prio_cols = ['patient_id', 'disc', 'Pfirrmann']
    for col in reversed(prio_cols):
        cols.insert(0, cols.pop(cols.index(col)))
    df_final = df_final[cols]

    # Guardar
    df_final.to_csv(OUTPUT_CSV, index=False)
    print(f"\nProceso completado con éxito.")
    print(f"Total de registros: {len(df_final)}")
    print(f"Archivo guardado y ordenado en: {OUTPUT_CSV}")

if __name__ == "__main__":
    combine_radiomics_and_labels()

Cargado disco 1 (L5-S1)
Cargado disco 2 (L4-L5)
Cargado disco 3 (L3-L4)
Cargado disco 4 (L2-L3)
Cargado disco 5 (L1-L2)

Proceso completado con éxito.
Total de registros: 5187
Archivo guardado y ordenado en: /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/radiomics_final_with_labels.csv
